# Weak labels analysis

Human biome and EFG labels vs. LLM predictions from `l4_weak_labels_010726_high`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

repo_root = Path.cwd().resolve()
for candidate in (repo_root, *repo_root.parents):
    if (candidate / "data").exists():
        repo_root = candidate
        break
else:
    raise FileNotFoundError("Could not locate repository root containing data/.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from evals_local.loaders import load_predictions
from evals_local.metrics import label_metrics, record_confusion
from evals_local.normalizers import to_label_set

In [2]:
RUN = "l4_weak_labels_010726_high"
SHEET = "l4_biome_efg"

truth_candidates = [
    repo_root / "data/consistency-check-datasets/data-coding/weak_labels_dataset.xlsx",
    repo_root / "data/consistency-check-datasets/data-coding/weak-labels/biome-efg-labels.xlsx",
]
TRUTH_PATH = next((path for path in truth_candidates if path.exists()), None)
if TRUTH_PATH is None:
    raise FileNotFoundError("Could not find the weak-label truth workbook.")

TRUTH_PATH

PosixPath('/Users/hunain/d/coding-projects/biodiversity/data/consistency-check-datasets/data-coding/weak-labels/biome-efg-labels.xlsx')

## Load and join

In [3]:
human = pd.read_excel(TRUTH_PATH, sheet_name=SHEET).rename(columns={"UT (Unique WOS ID)": "custom_id"})
human["custom_id"] = human["custom_id"].astype(str).str.strip()

biome_preds = load_predictions("ecosystems_biome", RUN)[["custom_id", "pred_biome"]].copy()
efg_preds = load_predictions("ecosystems_efg", RUN)[["custom_id", "pred_efg"]].copy()

for preds in (biome_preds, efg_preds):
    preds["custom_id"] = preds["custom_id"].astype(str).str.strip()

df = (
    human.merge(biome_preds.drop_duplicates("custom_id"), on="custom_id", how="left", validate="one_to_one")
    .merge(efg_preds.drop_duplicates("custom_id"), on="custom_id", how="left", validate="one_to_one")
)

summary = pd.DataFrame(
    [
        {"level": "biome", "truth_rows": human["biome"].notna().sum(), "prediction_rows": biome_preds["custom_id"].nunique(), "joined_predictions": df["pred_biome"].notna().sum()},
        {"level": "efg", "truth_rows": human["efg"].notna().sum(), "prediction_rows": efg_preds["custom_id"].nunique(), "joined_predictions": df["pred_efg"].notna().sum()},
    ]
)
summary

,level,truth_rows,prediction_rows,joined_predictions
0,biome,50,47,47
1,efg,50,46,46


In [4]:
df[["custom_id", "biome", "pred_biome", "efg", "pred_efg"]].head()

,custom_id,biome,pred_biome,efg,pred_efg
0,WOS:000232993900006,"[""Deserts and semi-deserts""]",[Deserts and semi-deserts],"[""Thorny deserts and semi-deserts""]",[Thorny deserts and semi-deserts]
1,WOS:000293173900014,"[""Savannas and grasslands""]",[Temperate-boreal forests & woodlands],"[""Pyric tussock savannas""]",[Boreal and temperate high montane forests and...
2,WOS:000290171900002,"[""Savannas and grasslands""]",[Savannas and grasslands],"[""Pyric tussock savannas""]",[Trophic savannas]
3,WOS:000265939700017,"[""Savannas and grasslands""]",[Savannas and grasslands],"[""Trophic savannas""]",[Trophic savannas]
4,WOS:000294170500002,"[""Tropical-subtropical forests""]",[Tropical-subtropical forests],"[""Tropical-subtropical lowland rainforests""]",[Tropical-subtropical lowland rainforests]


## Metrics

In [5]:
biome_scored = record_confusion(df.copy(), "biome", "pred_biome", base="biome", normalizer=to_label_set)
efg_scored = record_confusion(df.copy(), "efg", "pred_efg", base="efg", normalizer=to_label_set)

biome_metrics = label_metrics(df, "biome", "pred_biome", normalizer=to_label_set)
efg_metrics = label_metrics(df, "efg", "pred_efg", normalizer=to_label_set)

weighted_kpis = pd.concat(
    [
        biome_metrics.assign(level="biome"),
        efg_metrics.assign(level="efg"),
    ],
    ignore_index=True,
).query("label in ['_macro', '_weighted', '_micro']")

weighted_kpis[["level", "label", "n", "precision", "recall", "f1", "kappa", "support"]]

,level,label,n,precision,recall,f1,kappa,support
6,biome,_macro,50,0.444444,0.429630,0.436693,0.386254,51
7,biome,_weighted,50,0.941176,0.862745,0.900137,0.650629,51
8,biome,_micro,300,0.936170,0.862745,0.897959,0.878079,51
19,efg,_macro,50,0.418333,0.418182,0.391270,0.364050,51
20,efg,_weighted,50,0.839216,0.745098,0.767507,0.638833,51
21,efg,_micro,500,0.826087,0.745098,0.783505,0.760318,51


## Mismatches

In [8]:
df["biome_match"] = df.apply(lambda r: to_label_set(r["biome"]) == to_label_set(r["pred_biome"]), axis=1)
df["efg_match"] = df.apply(lambda r: to_label_set(r["efg"]) == to_label_set(r["pred_efg"]), axis=1)

df.loc[
    ~(df["biome_match"] & df["efg_match"]),
    ["custom_id", "Article Title", "biome", "pred_biome", "efg", "pred_efg", "biome_match", "efg_match"],
].head(20)

,custom_id,Article Title,biome,pred_biome,efg,pred_efg,biome_match,efg_match
1,WOS:000293173900014,Estimating population size and habitat suitabi...,"[""Savannas and grasslands""]",[Temperate-boreal forests & woodlands],"[""Pyric tussock savannas""]",[Boreal and temperate high montane forests and...,False,False
2,WOS:000290171900002,Distribution of savannah elephants (Loxodonta ...,"[""Savannas and grasslands""]",[Savannas and grasslands],"[""Pyric tussock savannas""]",[Trophic savannas],True,False
5,WOS:000269692700008,Bushmeat hunting in Dwesa and Cwebe Nature Res...,"[""Tropical-subtropical forests""]",[Temperate-boreal forests & woodlands],"[""Tropical-subtropical dry forests and thickets""]",[Oceanic cool temperate rainforests],False,False
15,WOS:A1997YK80600014,Impact of hunting on large vertebrates in the ...,"[""Tropical-subtropical forests""]",[Tropical-subtropical forests],"[""Tropical-subtropical dry forests and thickets""]",[Tropical-subtropical lowland rainforests],True,False
22,WOS:000291040800011,Large vertebrate responses to forest cover and...,"[""Tropical-subtropical forests""]",[Tropical-subtropical forests],"[""Tropical-subtropical dry forests and thickets""]",[Tropical-subtropical lowland rainforests],True,False
25,WOS:000220627900003,Human impact on wildlife populations within a ...,"[""Tropical-subtropical forests""]",[Tropical-subtropical forests],"[""Tropical-subtropical lowland rainforests""]",NaN,True,False
31,WOS:000266752500026,Discriminating between village and commercial ...,"[""Tropical-subtropical forests""]",[Tropical-subtropical forests],"[""Tropical-subtropical lowland rainforests""]",[Tropical-subtropical montane rainforests],True,False
33,WOS:000227928800004,Effects of habitat disturbance and hunting on ...,"[""Tropical-subtropical forests""]",[Tropical-subtropical forests],"[""Tropical-subtropical lowland rainforests""]",[Tropical-subtropical montane rainforests],True,False
38,WOS:000323101700011,Use of mineral licks by mammals and birds in h...,"[""Tropical-subtropical forests""]",NaN,"[""Tropical-subtropical lowland rainforests""]",NaN,False,False
39,WOS:000260556500005,Habitat selection and the change in distributi...,"[""Tropical-subtropical forests""]",NaN,"[""Tropical-subtropical lowland rainforests""]",NaN,False,False
